# 04 · 근거를 달고 답하는 질의응답 · 진단
**3세션 · 고르기 · RAG (오후)** — 실습 30분

| 순서 | 할 일 |
|---|---|
| 1 청킹 · 2 색인 | 03_rag_index 노트북에서 한 것 — 여기서 다시 만든다 |
| 3 ask() | 검색 → 주입 → 호출, 출처 각주 포함 |
| 4 거절 확인 | 답이 없는 질문 3개에 "자료에 없음"이 나오는가 — **건너뛰지 않는다** |
| 5 10문항 채점 | 답있음 5 · 답없음 3 · 바꿔쓴 2 |

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # day 폴더로 이동 — data/ · prompts/ · src/ 가 여기 있다

from common import llm      # 공용 API 도구 (ETRI_LLM_3days/common/llm.py)
if llm.MOCK:
    print("모델:", llm.MODEL, "| 모의 모드")
else:
    print("모델:", llm.MODEL, "| 실제 호출")

from src import rag
if rag.EMBED == "hash":
    print("임베딩: 간이 해시 (EMBED=hash)")
else:
    print("임베딩:", rag.ST_MODEL)
from src import qa

chunks = rag.chunk_folder("data/kb")
idx = rag.Index(chunks)
print("조각", len(chunks))

## 3. ask() — 검색 · 주입 · 호출
자료마다 번호와 출처를 달아 지시와 분리합니다. 규칙은 맨 앞, 질문은 맨 끝 — 어제 배운 위치.

In [ ]:
print(rag.RULES)
answer, hits = rag.ask("휴가 신청은 어떻게 해요?", idx, k=3)
print()
print(answer)

# 답에 붙은 각주 번호가 어느 파일을 가리키는지
footnotes = rag.cited(answer)
cited_files = []
for n in footnotes:
    if n <= len(hits):
        cited_files.append(hits[n - 1].source)     # 각주 [1]은 hits[0]
print()
print("각주:", footnotes, "→", cited_files)

최종 프롬프트를 눈으로 봅니다. 모델이 본 것은 이것이 전부입니다.

In [ ]:
print(rag.LAST_PROMPT)

## 4. 거절 확인
지어낸 금액을 말하지 않는 것이 이 시스템의 **가장 중요한 동작**입니다.

In [ ]:
NO_ANSWER = ["올해 성과급 지급률은 몇 퍼센트인가요?",
             "사내 헬스장은 몇 시까지 운영하나요?",
             "육아휴직 기간에 급여는 얼마나 나오나요?"]
for q in NO_ANSWER:
    answer, hits = rag.ask(q, idx)
    print(f"Q {q}")
    print(f"A {answer.strip()}")
    print()

> 세 번째 질문은 토론 거리입니다. 일반 업무 규칙 제18조는 "처우는 관계 법령에 따른다"고만 합니다. 이것은 *답*일까요, *자료에 없음*일까요? 테스트 문항으로 쓰려면 기준을 먼저 정해야 합니다.

## 5. 10문항 채점
어제와 같은 방식 — 고정 문항(`data/qa_tests.jsonl`) · 판정 기준 · 반복 기록.
다른 점은 실패를 **어느 단계의 고장인지**로 나눈다는 것입니다.
코드는 위치를 좁혀 줄 뿐입니다. *검색*인지 *청킹*인지는 `debug()`로 조각을 열어 사람이 가립니다 — 정답 문서가 아예 안 왔으면 검색, 왔는데 답이 잘려 있으면 청킹.

| 고장 | 판정 | 고치는 곳 |
|---|---|---|
| 검색 · 청킹 | 넣은 자료에 답이 든 조각이 없다 | 검색: k · 질문 말투 · 키워드 검색 / 청킹: 구조 경계 · 겹침 |
| 있는데 못 씀 | 자료에 답이 있는데 답에 없다 | 주입 코드 · 프롬프트 |
| 지어냄 | 답 없는 질문에 "자료에 없음"이 아니다 | 거절 지시 |
| 출처 오류 | 답은 맞는데 각주가 다른 문서를 가리킨다 | 각주 규칙 |

In [ ]:
passed, fails, rows = qa.run_qa(idx, k=3)

## 6. 진단 — 답만 보지 말고 중간을 찍는다

In [ ]:
# 처음 실패한 문항을 연다. 실패가 없으면 아래 질문으로.
FAIL_Q = "휴가 쓰려면 며칠 전에 말해야 해요?"
for r in rows:
    if not r["ok"]:
        FAIL_Q = r["q"]
        break

answer, hits = rag.debug(FAIL_Q, idx, k=3)